# 01 — EDA: ERCOT Coast Zone Load vs Houston Weather

Goal of this notebook: look at the raw hourly load series, decompose it into
trend/weekly seasonality/residual, and confirm the U-shaped relationship
between temperature and demand (rises at both heating and cooling extremes).

This is exploratory only, no modeling here yet. Run `src/data_pipeline.py`
first if `data/processed/load_weather_joined.parquet` doesn't exist yet.

In [ ]:
# lets src/ be imported when running this notebook from the notebooks/ folder
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.seasonal import STL

from src import config

df = pd.read_parquet(config.JOINED_DATA_PATH).asfreq("h")
print(df.shape)
print(df.index.min(), "to", df.index.max())
df.head()

In [ ]:
df.describe()

## Raw hourly load over the full range

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df["load_mw"], linewidth=0.4)
ax.set_title("Hourly Coast zone load")
ax.set_ylabel("MW")
plt.show()

## Seasonal decomposition

Using weekly seasonality (period = 24 * 7 hours), per the project brief.
STL is used here instead of classic seasonal_decompose since it copes better
with a seasonal pattern that shifts a bit over the year (which load does,
weekday/weekend gaps are not identical in summer vs winter).

In [ ]:
# STL can't handle NaN inside its local windows — a single missing hour turns into
# a much wider NaN stretch in the output. Fine to fully interpolate for this plot,
# this is just a visual check, not a modeling input.
load_for_stl = df["load_mw"].interpolate(method="time")

stl = STL(load_for_stl, period=24 * 7, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(df.index, df["load_mw"], linewidth=0.4)
axes[0].set_title("Observed")
axes[1].plot(df.index, result.trend, linewidth=0.6)
axes[1].set_title("Trend")
axes[2].plot(df.index, result.seasonal, linewidth=0.3)
axes[2].set_title("Weekly seasonal")
axes[3].plot(df.index, result.resid, linewidth=0.3)
axes[3].set_title("Residual")
plt.tight_layout()
plt.show()

## Load vs temperature — checking for the U-shape

The core hypothesis this project rests on: demand should rise at both cold
and hot extremes (heating and cooling load), not just trend up with heat.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df["temp_c"], df["load_mw"], alpha=0.03, s=5)

# binned average makes the U-shape visible through the scatter noise
bins = pd.cut(df["temp_c"], bins=30)
binned_avg = df.groupby(bins, observed=True)["load_mw"].mean()
bin_centers = [interval.mid for interval in binned_avg.index]
ax.plot(bin_centers, binned_avg.values, color="red", linewidth=2, label="binned average")

ax.set_xlabel("Temperature (C)")
ax.set_ylabel("Load (MW)")
ax.set_title("Load vs temperature")
ax.legend()
plt.show()

**Note:** if the binned average line doesn't show a clean U-shape, remember
the known simplification from the brief — Coast zone load includes a lot of
temperature-insensitive industrial demand, so expect more baseline noise than
a purely residential zone would have. The U-shape may be present but shallow.